# Gold layer - route KPI window

Build route-level KPI records from Silver transit metrics.
This step pivots metric-level Silver outputs into a wide Gold table
and attaches a simple data-quality flag for downstream analysis.

In [0]:
_ = spark.sql("USE azure_streaming_mvp")

### Gold KPI construction

Create a window-level Gold table by pivoting `silver_transit_metrics`
into route-level KPI records.

In [0]:
%sql
-- Gold: route KPIs per window
CREATE OR REPLACE TABLE gold_route_kpi_window
USING DELTA
AS
WITH s AS (
  SELECT *
  FROM silver_transit_metrics
),
wide AS (
  SELECT 
    window_start,
    window_end,
    route_id,
    
    -- KPI metrics
    MAX(CASE WHEN metric = 'delay_sec' THEN avg_value END) AS avg_delay_sec,
    MAX(CASE WHEN metric = 'occupancy' THEN avg_value END) AS avg_occupancy_pct,

    -- Event counts
    MAX(CASE WHEN metric = 'delay_sec' THEN n_events END) AS n_events_delay,
    MAX(CASE WHEN metric = 'occupancy' THEN n_events END) AS n_events_occupancy,

    -- Latency signals (based on delay metric)
    MAX(CASE WHEN metric = 'delay_sec' THEN late_event_rate END) AS late_rate_delay,
    MAX(CASE WHEN metric = 'delay_sec' THEN avg_ingest_delay_sec END) AS avg_ingest_delay_sec,
    MAX(CASE WHEN metric = 'delay_sec' THEN n_clock_skew END) AS n_clock_skew
        
  FROM s
  GROUP BY window_start, window_end, route_id
)
SELECT
  *,
  CASE
    WHEN COALESCE(n_clock_skew, 0.0) > 0 THEN 'CLOCK_SKEW'
    WHEN COALESCE(n_events_delay, 0) < 5 THEN 'LOW_VOLUME'
    WHEN COALESCE(late_rate_delay, 0.0) > 0.30 THEN 'HIGH_LATE_RATE'
    ELSE 'OK'
  END AS dq_flag
FROM wide;

num_affected_rows,num_inserted_rows


## Data validation checks

These checks confirm that the Gold route KPI table was built successfully
and summarize the current data-quality flag distribution.

In [0]:
%sql
SELECT metric, value
FROM (

    SELECT
        1 AS sort_order, 
        'row_count' AS metric,
        CAST(COUNT(*) AS STRING) AS value
    FROM gold_route_kpi_window

    UNION ALL

    SELECT 
        2 AS sort_order,
        'latest_window_end' AS metric,
        CAST(MAX(window_end) AS STRING) AS value
    FROM gold_route_kpi_window

    UNION ALL

    SELECT
        3 AS sort_order, 
        CONCAT('dq_flag=', dq_flag) AS metric,
        CAST(COUNT(*) AS STRING) AS value
    FROM gold_route_kpi_window
    GROUP BY dq_flag

)
ORDER BY sort_order;

metric,value
row_count,86
latest_window_end,2026-03-08 15:45:00
dq_flag=LOW_VOLUME,86


### Preview Gold route KPIs

Inspect recent route-level KPI records from the Gold layer.

In [0]:
%sql
SELECT *
FROM gold_route_kpi_window
ORDER BY window_start DESC, route_id
LIMIT 20;

window_start,window_end,route_id,avg_delay_sec,avg_occupancy_pct,n_events_delay,n_events_occupancy,late_rate_delay,avg_ingest_delay_sec,n_clock_skew,dq_flag
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,B1,null,44.25,null,4,null,null,null,LOW_VOLUME
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,M1,465.5,16.0,4,1,0.75,196.75,0,LOW_VOLUME
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,M2,null,78.0,null,1,null,null,null,LOW_VOLUME
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,R10,null,5.0,null,1,null,null,null,LOW_VOLUME
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,T1,258.6666666666667,1.0,3,1,0.3333333333333333,155.33333333333334,0,LOW_VOLUME
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,X3,null,20.0,null,1,null,null,null,LOW_VOLUME
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,B2,57.0,65.66666666666667,1,3,1.0,306.0,0,LOW_VOLUME
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,M1,585.0,2.0,1,1,1.0,552.0,0,LOW_VOLUME
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,M2,null,58.5,null,2,null,null,null,LOW_VOLUME
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,R10,76.0,null,1,null,1.0,442.0,0,LOW_VOLUME


In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")